# Data modeling 5: normalized and operational models

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/data-modeling/data-modeling-05-normalized-and-operational-models.ipynb)

Analytics data starts in operational systems: the order application, the CRM, the billing service. Those
databases are modeled for a different job, recording one business event at a time, quickly and without
contradictions. The tool for that is **normalization**. Knowing how and why operational schemas are normalized
lets you read source systems correctly, and explains why the warehouse deliberately does the opposite.

This lesson normalizes a spreadsheet export step by step, enforces the result with constraints, and then looks
at the two common alternatives to flat normalized tables: denormalized stars and nested records.

**Prerequisites:** [Data modeling 1: grain, facts and dimensions](data-modeling-01-grain-facts-and-dimensions.ipynb). Lessons 2 to 4 help but are not required.

*Last reviewed: November 2026, against DuckDB 2.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q duckdb~=2.0

## Setup and sample data

The next cell creates an in-memory DuckDB database and loads `src_customers` (the CRM's current view of each customer), `src_products` (the product catalogue), `src_orders` (one row per order, with fulfilment timestamps) and `src_order_lines` (one row per product on an order): extracts from a small online shop that sells coffee, tea and kitchenware. Every lesson starts from these source tables; [Data modeling 0](data-modeling-00-start-here.ipynb) describes them.

In [ ]:
import duckdb

con = duckdb.connect()  # an in-memory database: the warehouse for this notebook
con.sql("""
create table src_customers (
    customer_id int primary key, name varchar, email varchar, city varchar, segment varchar, created_on date
);
insert into src_customers values
    (101, 'Acme Corp',     'ops@acme.example',     'Leeds', 'smb',        '2025-11-02'),
    (102, 'Bright Retail', 'buy@bright.example',   'York',  'enterprise', '2025-11-20'),  -- was smb until Feb 1
    (103, 'Cyan Traders',  'hi@cyan.example',      'Bath',  'enterprise', '2025-12-05'),
    (104, 'Delta Grocers', 'orders@delta.example', 'Leeds', 'smb',        '2026-01-08'),  -- moved from Hull on Feb 15
    (105, 'Echo Supplies', null,                   'Derby', 'smb',        '2026-01-19'),
    (106, 'Foxtrot Foods', 'team@foxtrot.example', 'York',  'mid-market', '2026-02-10');  -- has not ordered yet
create table src_products (
    sku varchar primary key, product_name varchar, category varchar, department varchar, list_price decimal(8, 2)
);
insert into src_products values
    ('SKU-01', 'House Blend 1kg',    'Coffee',  'Grocery', 18.00),
    ('SKU-02', 'Espresso Beans 1kg', 'Coffee',  'Grocery', 22.00),
    ('SKU-03', 'Green Tea 100 bags', 'Tea',     'Grocery',  6.50),
    ('SKU-04', 'Earl Grey 100 bags', 'Tea',     'Grocery',  5.50),
    ('SKU-05', 'Stoneware Mug',      'Mugs',    'Kitchen',  9.00),
    ('SKU-06', 'Travel Mug',         'Mugs',    'Kitchen', 15.00),
    ('SKU-07', 'Gooseneck Kettle',   'Kettles', 'Kitchen', 65.00),
    ('SKU-08', 'Electric Kettle',    'Kettles', 'Kitchen', 45.00);
create table src_orders (
    order_id int primary key, customer_id int, order_ts timestamp, channel varchar, status varchar,
    shipping_fee decimal(8, 2), shipped_ts timestamp, delivered_ts timestamp
);
insert into src_orders values
    (1001, 101, '2026-01-05 09:12', 'web',   'delivered', 4.99, '2026-01-06 10:00', '2026-01-08 15:00'),
    (1002, 102, '2026-01-09 14:30', 'store', 'delivered', 0.00, '2026-01-09 14:30', '2026-01-09 14:30'),
    (1003, 103, '2026-01-14 11:05', 'web',   'delivered', 0.00, '2026-01-15 09:00', '2026-01-19 12:00'),
    (1004, 101, '2026-01-28 16:40', 'web',   'delivered', 4.99, '2026-01-29 11:00', '2026-02-02 10:30'),
    (1005, 104, '2026-02-03 10:20', 'web',   'delivered', 4.99, '2026-02-04 08:45', '2026-02-06 16:10'),
    (1006, 102, '2026-02-11 13:15', 'web',   'shipped',   0.00, '2026-02-12 09:30', null),
    (1007, 105, '2026-02-14 09:50', 'store', 'delivered', 0.00, '2026-02-14 09:50', '2026-02-14 09:50'),
    (1008, 104, '2026-02-20 18:05', 'web',   'placed',    4.99, null,               null),
    (1009, 103, '2026-02-24 12:00', 'web',   'cancelled', 4.99, null,               null),
    (1010, 101, '2026-02-27 08:30', 'store', 'delivered', 0.00, '2026-02-27 08:30', '2026-02-27 08:30');
create table src_order_lines (
    order_id int, line_no int, sku varchar, qty int, unit_price decimal(8, 2), discount decimal(8, 2),
    primary key (order_id, line_no)
);
insert into src_order_lines values
    (1001, 1, 'SKU-01',  2, 18.00,  0.00), (1001, 2, 'SKU-05', 4,  9.00, 0.00),
    (1002, 1, 'SKU-07',  1, 65.00,  5.00),
    (1003, 1, 'SKU-02', 10, 22.00, 20.00), (1003, 2, 'SKU-03', 5,  6.50, 0.00),
    (1004, 1, 'SKU-06',  2, 15.00,  0.00),
    (1005, 1, 'SKU-01',  3, 18.00,  0.00), (1005, 2, 'SKU-04', 2,  5.50, 0.00), (1005, 3, 'SKU-08', 1, 45.00, 0.00),
    (1006, 1, 'SKU-02', 20, 22.00, 40.00),
    (1007, 1, 'SKU-05',  6,  9.00,  0.00),
    (1008, 1, 'SKU-03',  4,  6.50,  0.00),
    (1009, 1, 'SKU-07',  1, 65.00,  0.00),
    (1010, 1, 'SKU-01',  1, 18.00,  0.00), (1010, 2, 'SKU-06', 1, 15.00, 0.00);
""")
con.sql("show tables").show()

## Two jobs, two shapes

| | Operational (OLTP) | Analytical (OLAP) |
|---|---|---|
| Typical work | place an order, update an address | revenue by month and segment |
| Rows touched per query | a few, by key | millions, by scan |
| Model | normalized: each fact stored once | dimensional: denormalized stars |
| Optimized for | correct, concurrent writes | fast, simple reads |
| Storage | row-oriented, indexed (Postgres, MySQL) | columnar, compressed (Snowflake, BigQuery, DuckDB) |
| History | usually current state only | history is the point |

Modeling also happens at three levels of detail. A **conceptual** model names the business entities and how
they relate (customers place orders; orders contain products), for agreeing on meaning with the business. A
**logical** model adds every attribute, key and cardinality, independent of any database. A **physical**
model is the actual tables, types, constraints, indexes and partitions for one engine. Normalization is a
logical-model activity; the physical model may then denormalize for performance.

**Where it matters**
- Reading a new source system, designing an application database, and explaining why the warehouse looks different.

**Watch out for**
- Running heavy analytics directly on the operational database: scans compete with the writes the business depends on.

## Normalization by example

Normalization removes redundancy so that each fact is stored exactly once. The normal forms are steps:

- **First normal form (1NF):** every column holds one value; no lists in a cell and no repeating column groups
  (`product_1`, `product_2`, ...).
- **Second normal form (2NF):** 1NF, and every non-key column depends on the *whole* key. In a table keyed by
  `(order_id, line_no)`, the customer depends only on `order_id`, so it belongs in an orders table.
- **Third normal form (3NF):** 2NF, and non-key columns depend only on the key, not on other non-key columns.
  A customer's city depends on the customer, not on the order, so it belongs in a customers table.

A compact way to remember 3NF: every non-key column depends on "the key, the whole key, and nothing but the
key". Higher forms (BCNF, 4NF, 5NF) deal with rarer overlapping-key and multi-valued cases.

Why bother? Redundant copies drift apart. These are the **anomalies**: an *update anomaly* (change a customer's
city on one order row but not the others), an *insert anomaly* (you cannot record a new product until someone
orders it) and a *delete anomaly* (deleting the only order for a product loses the product).

**Where it matters**
- Application databases, master data, and the clean core layer of an Inmon-style warehouse (lesson 6).

**Watch out for**
- Normalizing reference data so far that every query needs eight joins: stop at 3NF unless you have a reason.

### Example

The shop's first system was a spreadsheet. Each row is an order, the products are packed into one cell, and the
customer's details are repeated on every order. One clerk has already typed a new city into one row.

In [ ]:
con.sql("""
create table order_sheet (order_id int, order_date date, customer varchar, customer_city varchar, items varchar);
insert into order_sheet values
    (1001, '2026-01-05', 'Acme Corp',     'Leeds', 'SKU-01:House Blend 1kg:2; SKU-05:Stoneware Mug:4'),
    (1004, '2026-01-28', 'Acme Corp',     'Leeds', 'SKU-06:Travel Mug:2'),
    (1010, '2026-02-27', 'Acme Corp',     'Wakefield', 'SKU-01:House Blend 1kg:1; SKU-06:Travel Mug:1'),
    (1005, '2026-02-03', 'Delta Grocers', 'Hull',  'SKU-01:House Blend:3; SKU-04:Earl Grey 100 bags:2');
""")
con.sql("select * from order_sheet order by order_id").show()

In [ ]:
con.sql("""
-- the update anomaly: one customer, two cities
select customer, list(distinct customer_city) as cities_on_file from order_sheet group by customer order by customer
""").show()

**What to notice:** Acme Corp lives in Leeds on two rows and Wakefield on a third, and no rule says which is right. Product SKU-01 is
called "House Blend 1kg" on some rows and "House Blend" on another. The `items` cell holds a list, so "how many
mugs did we sell" needs string parsing. Each problem comes from storing a fact more than once or packing several
facts into one cell.

### Example

Step 1 (1NF): one value per cell. Split the item list into one row per product. Steps 2 and 3 (2NF, 3NF): move
product and customer attributes into tables of their own, keyed by what they depend on.

In [ ]:
con.sql("""
create table sheet_lines as                       -- 1NF: one row per order line, one value per column
select o.order_id, o.order_date, o.customer, o.customer_city,
       generate_subscripts(string_split(o.items, '; '), 1)       as line_no,
       split_part(unnest(string_split(o.items, '; ')), ':', 1)   as sku,
       split_part(unnest(string_split(o.items, '; ')), ':', 2)   as product_name,
       cast(split_part(unnest(string_split(o.items, '; ')), ':', 3) as int) as qty
from order_sheet o;
""")
con.sql("select * from sheet_lines order by order_id, line_no").show()

In [ ]:
con.sql("""
-- 2NF: product_name depends on sku alone, and customer and date on order_id alone, not on (order_id, line_no)
create table n_products as select sku, max(product_name) as product_name from sheet_lines group by sku;
create table n_lines    as select order_id, line_no, sku, qty from sheet_lines;
-- 3NF: city depends on the customer, not the order
create table n_customers as
select row_number() over (order by customer) as customer_id, customer as name,
       arg_max(customer_city, order_date) as city                  -- a rule is needed to pick one city
from sheet_lines group by customer;
create table n_orders as
select distinct s.order_id, s.order_date, c.customer_id from sheet_lines s join n_customers c on c.name = s.customer;
""")
for table in ["n_customers", "n_orders", "n_products", "n_lines"]:
    con.sql(f"select * from {table} order by all").show()

**What to notice:** The 1NF table has one row per product on an order (7 rows from 4 orders), so quantities can be summed. After 2NF
and 3NF each fact lives in one place: each customer's city is a single value in `n_customers` (the rule picked the
most recent, Wakefield) and each product's name a single value in `n_products`. Normalizing did not decide which
of the conflicting values was right; it made the conflict impossible from now on. Picking "House Blend 1kg" with
`max()` was a guess, which is why cleanup like this needs a business owner.

## Constraints enforce the model

A normalized schema is only as good as its enforcement. Constraints make the database reject rows that break
the model:

- `primary key`: the grain, unique and not null;
- `foreign key`: every reference points at an existing row (no orphan order lines);
- `not null`, `check`: required values and allowed ranges (`qty > 0`);
- `unique`: alternate keys, such as a customer's email.

Operational databases (Postgres, MySQL, DuckDB) enforce all of these. Most cloud warehouses do **not**:
Snowflake (standard tables), BigQuery, Redshift and Databricks accept primary and foreign key declarations
but enforce only `not null` (Databricks also enforces `check`). The others are *informational*: the optimizer may trust them,
and duplicates load without complaint. In a warehouse, tests (lesson 9) do the enforcing.

**Where it matters**
- Application schemas, staging tables that should reject bad input, and documenting intent in warehouse DDL.

**Watch out for**
- Declaring a primary key in Snowflake or BigQuery and assuming duplicates are impossible: they are not.

### Example

Declare the normalized tables with constraints, then try to insert rows that break each rule.

In [ ]:
con.sql("""
create table customers (customer_id int primary key, name varchar not null, email varchar unique);
create table products  (sku varchar primary key, product_name varchar not null, list_price decimal(8, 2) check (list_price > 0));
create table orders    (order_id int primary key, customer_id int not null references customers (customer_id), order_ts timestamp not null);
create table order_lines (
    order_id int references orders (order_id), line_no int, sku varchar references products (sku),
    qty int check (qty > 0), primary key (order_id, line_no)
);
insert into customers select customer_id, name, email from src_customers;
insert into products  select sku, product_name, list_price from src_products;
insert into orders    select order_id, customer_id, order_ts from src_orders;
insert into order_lines select order_id, line_no, sku, qty from src_order_lines;
""")

bad_rows = {
    "duplicate order line": "insert into order_lines values (1001, 1, 'SKU-02', 1)",
    "line for a missing order": "insert into order_lines values (9999, 1, 'SKU-01', 1)",
    "unknown product": "insert into order_lines values (1002, 2, 'SKU-99', 1)",
    "zero quantity": "insert into order_lines values (1002, 2, 'SKU-01', 0)",
    "duplicate email": "insert into customers values (200, 'Copycat Ltd', 'ops@acme.example')",
}
for label, statement in bad_rows.items():
    try:
        con.sql(statement)
        print(f"accepted  {label}")
    except Exception as error:  # DuckDB raises a ConstraintException for each of these
        print(f"rejected  {label}: {str(error).splitlines()[0].removeprefix('Constraint Error: ')}")

**What to notice:** Every bad row was rejected with a constraint error, and nothing was written. In a warehouse that does not enforce
keys, all five would have loaded silently: a duplicate line that doubles revenue, an orphan line no join can
place, and a zero-quantity line. That is the gap model tests exist to close.

## Why the warehouse denormalizes

A star schema is denormalized on purpose. `dim_product` repeats "Kitchen" on every kitchen product, which 3NF
forbids, because analytics trades the write-side guarantees it does not need for reads it does:

- Warehouse tables are written by controlled batch loads, not by thousands of concurrent users, so the update
  anomalies normalization prevents are handled by rebuilding the dimension from its source.
- Every join is work at query time, and analytical queries touch most of the table.
- Columnar compression makes repeated values almost free to store.

The rule of thumb: **normalize where data is written, denormalize where it is read**, and keep the
normalized source (or a normalized core layer) so the denormalized tables can always be rebuilt.

**Where it matters**
- Deciding how far to flatten: stars for marts, normalized or Data Vault for the integration layer (lesson 6).

**Watch out for**
- Denormalizing a fast-changing attribute onto a huge fact table: every change rewrites millions of rows.

## Nested records

A third option sits between normalized tables and flat denormalization: store child rows *inside* the parent
row as a list of structs. BigQuery, Snowflake (`VARIANT`/`ARRAY`), Spark, Parquet and DuckDB all support it.
An order with its lines nested is one row; reading an order needs no join, and the lines cannot be separated
from the order or duplicated by a join.

Nesting suits children that always travel with the parent and are rarely queried on their own (the lines of an
order, the items in an event payload). It suits poorly children that are queried, updated or joined
independently, and many-to-many relationships. Queries unnest when they need one row per child.

**Where it matters**
- Event payloads and API responses, BigQuery tables that avoid large joins, document databases.

**Watch out for**
- Aggregating a parent column after unnesting: unnesting is a join in disguise, with the same fan-out risk.

### Example

Nest each order's lines into the order row, then unnest to count units by product.

In [ ]:
con.sql("""
create table orders_nested as
select o.order_id, o.order_ts, o.customer_id, o.shipping_fee,
       list({'sku': l.sku, 'qty': l.qty, 'unit_price': l.unit_price} order by l.line_no) as lines
from src_orders o join src_order_lines l using (order_id)
group by all
""")
con.sql("select order_id, shipping_fee, lines from orders_nested where order_id in (1001, 1005) order by order_id").show()

In [ ]:
con.sql("""
select line.sku, sum(line.qty) as units,
       sum(o.shipping_fee) as shipping_fee_summed_after_unnest   -- the fan trap again
from orders_nested o, unnest(o.lines) as t(line)
group by line.sku order by line.sku
""").show()

**What to notice:** Order 1005 is one row whose `lines` column holds three structs. Unnesting gives one row per line, so units per SKU
come out right. The last column shows the catch: `shipping_fee` is an order-level value, and after the unnest it
repeats on every line, so order 1005's fee is counted three times, under SKU-01, SKU-04 and SKU-08. Down the whole column the fees
add up to 39.92, against 24.95 actually charged. The
grain rules of lessons 1 and 4 apply to nested data exactly as they do to joins.

## Modeling for key-value and document stores

Databases such as DynamoDB, Cassandra and MongoDB turn the process around: you list the **access patterns** first
("get an order with its lines by order ID", "list a customer's orders by date") and shape the data so each pattern
is a single lookup, duplicating data where needed. There are no joins to fall back on, so a pattern you did not plan
for may need a new copy of the data. That is the opposite of normalization's "store each fact once", and it is
why these systems feed a warehouse rather than replace one. See [DynamoDB](../dynamodb/dynamodb.ipynb) for
single-table design and [MongoDB](../mongodb/mongodb.ipynb) for document modeling.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** A table keyed by `(order_id, line_no)` has a `customer_email` column. Which normal form does it break, and how do you fix it?

**A.** Second normal form: the email depends on the customer (through `order_id`), not on the whole key. Move the customer to the orders table, and the email to a customers table (which also satisfies 3NF).

**Q2.** What is an update anomaly? Give an example from the order spreadsheet.

**A.** Inconsistency created by changing one copy of a repeated fact. Acme Corp's city was changed on one order row, so the sheet now says Acme is in both Leeds and Wakefield.

**Q3.** You declared a primary key on a Snowflake table. Can duplicate keys still be loaded?

**A.** Yes. Snowflake (like BigQuery and Redshift) does not enforce primary or unique keys. Add a uniqueness test to the pipeline.

**Q4.** Why is it acceptable for `dim_product` to repeat the department name on every product?

**A.** The warehouse is loaded by controlled batch jobs that rebuild the dimension from its normalized source, so the anomalies normalization prevents cannot accumulate, and the flat table makes every query simpler and faster.

**Q5.** When would you nest order lines inside orders instead of using an order-line table?

**A.** When lines are almost always read with their order and rarely on their own: event payloads, BigQuery tables that avoid a large join. Use a separate table when lines are queried, updated or joined independently.

## Key takeaways

- Operational schemas are normalized for correct concurrent writes; warehouses denormalize for reads.
- 1NF: one value per cell. 2NF: depend on the whole key. 3NF: depend on nothing but the key.
- Normalization prevents update, insert and delete anomalies by storing each fact once.
- Operational databases enforce keys and constraints; most cloud warehouses do not, so tests must.
- Normalize where data is written, denormalize where it is read, and keep the normalized source to rebuild from.
- Nested records avoid joins for children that travel with their parent, but unnesting has the same fan-out risk as a join.

**Next:** Continue with [Data modeling 6: warehouse architectures](data-modeling-06-warehouse-architectures.ipynb).